## Section 1: Load Latest Pipeline Report

In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
import glob

# Get latest run report
PROJECT_ROOT = Path.cwd()
reports = sorted(glob.glob(str(PROJECT_ROOT / 'run_reports/run_report_*.json')))[-5:]
print(f"Found {len(reports)} recent reports:")
for r in reports:
    print(f"  - {Path(r).name}")

Found 5 recent reports:
  - run_report_20251223_153231.json
  - run_report_20251223_181529.json
  - run_report_20251223_183617.json
  - run_report_20251223_200719.json
  - run_report_20251224_205625.json


In [2]:
# Load the latest report
with open(reports[-1]) as f:
    report = json.load(f)

print(f"\n✅ Loaded: {Path(reports[-1]).name}")
print(f"\n📊 Report Summary:")
print(f"  - Run ID: {report['meta']['run_id']}")
print(f"  - Started: {report['meta']['started_utc']}")
print(f"  - Finished: {report['meta']['finished_utc']}")
print(f"  - Duration: {report['meta']['duration_seconds']:.0f} seconds ({report['meta']['duration_seconds']/60:.1f} minutes)")
print(f"  - Status: {report['meta']['overall_status']}")
print(f"  - Success: {report['meta']['success_count']}/{report['meta']['total_notebooks']}")


✅ Loaded: run_report_20251224_205625.json

📊 Report Summary:
  - Run ID: 20251224_205625
  - Started: 2025-12-24T20:56:25.212063
  - Finished: 2025-12-24T21:02:30.592636
  - Duration: 365 seconds (6.1 minutes)
  - Status: SUCCESS
  - Success: 3/26


## Section 2: Notebook-by-Notebook Results

In [3]:
# Create DataFrame of all notebooks
notebooks_data = []
for nb in report['notebooks']:
    notebooks_data.append({
        'Index': nb['index'],
        'Notebook': nb['name'],
        'Status': nb['status'],
        'Duration (s)': nb['duration_seconds'],
        'Has Error': 'Yes' if nb['error'] else 'No'
    })

df_notebooks = pd.DataFrame(notebooks_data)
print("\n📋 All Notebooks:")
print(df_notebooks.to_string(index=False))


📋 All Notebooks:
 Index                                                                                                                        Notebook  Status  Duration (s) Has Error
     0                          Pre-pilot – Spark & ADLS connectivity (_00_Pre_Pilot/Jupyter Notebooks/01_spark_adls_connectors.ipynb) SKIPPED      0.000000        No
     1                                  Bronze – data container & mounts (_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb) SKIPPED      0.000000        No
     2                                                      Bronze – raw CSV → Delta (_01_Bronze/Jupyter Notebooks/01_data_load.ipynb) SKIPPED      0.000000        No
     3                                              Silver – Policies (_02_Silver/Jupyter Notebooks/Policies/01_policies_silver.ipynb) SKIPPED      0.000000        No
     4                                                 Silver – Members (_02_Silver/Jupyter Notebooks/Members/02_members_silver.ipynb) SKIPPED     

## Section 3: Failed Notebooks

In [4]:
# Find failed notebooks
failed_notebooks = [nb for nb in report['notebooks'] if nb['status'] == 'FAILED']

if failed_notebooks:
    print(f"\n❌ {len(failed_notebooks)} FAILED NOTEBOOK(S):\n")
    for nb in failed_notebooks:
        print(f"Index: {nb['index']}")
        print(f"Name: {nb['name']}")
        print(f"Started: {nb['started_utc']}")
        print(f"Duration: {nb['duration_seconds']:.1f}s")
        print(f"\nError:")
        print(nb['error'])
        print("\n" + "="*80 + "\n")
else:
    print("\n✅ All notebooks executed successfully!")


✅ All notebooks executed successfully!


## Section 4: Error Analysis

In [5]:
# Extract error types
error_types = {}
for nb in report['notebooks']:
    if nb['error']:
        # Try to extract error type
        error_lines = nb['error'].split('\n')
        for line in error_lines:
            if 'Error' in line or 'Exception' in line:
                error_type = line.strip()
                error_types[error_type] = error_types.get(error_type, 0) + 1

if error_types:
    print("\n🔍 Error Types Encountered:")
    for error, count in sorted(error_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  ({count}x) {error}")
else:
    print("\n✅ No errors found!")


✅ No errors found!


## Section 5: Performance Analysis

In [6]:
# Analyze execution times
durations = [nb['duration_seconds'] for nb in report['notebooks']]

print("\n⏱️  Performance Metrics:")
print(f"  - Fastest: {min(durations):.1f}s")
print(f"  - Slowest: {max(durations):.1f}s")
print(f"  - Average: {sum(durations)/len(durations):.1f}s")
print(f"  - Total: {sum(durations):.0f}s ({sum(durations)/60:.1f}m)")

# Find slowest notebooks
slowest = sorted(report['notebooks'], key=lambda x: x['duration_seconds'], reverse=True)[:5]
print("\n🐌 Top 5 Slowest Notebooks:")
for i, nb in enumerate(slowest, 1):
    print(f"  {i}. {nb['duration_seconds']:.1f}s - {nb['name'][:60]}")


⏱️  Performance Metrics:
  - Fastest: 0.0s
  - Slowest: 137.6s
  - Average: 14.1s
  - Total: 365s (6.1m)

🐌 Top 5 Slowest Notebooks:
  1. 137.6s - ML – batch scoring claim fraud (_03_Gold/03_ML_Model_Trainin
  2. 122.3s - ML – batch scoring policy churn (_03_Gold/03_ML_Model_Traini
  3. 105.4s - ML – batch scoring high-cost claims (_03_Gold/03_ML_Model_Tr
  4. 0.0s - Pre-pilot – Spark & ADLS connectivity (_00_Pre_Pilot/Jupyter
  5. 0.0s - Bronze – data container & mounts (_01_Bronze/Jupyter Noteboo


## Section 6: How to Debug Notebook 20 (Policy Churn)

## Section 8: Compare Multiple Runs

In [9]:
# Compare last 3 runs
print(f"\n📊 Comparing Last {min(3, len(reports))} Runs:\n")

comparison_data = []
for report_file in reports[-3:]:
    with open(report_file) as f:
        r = json.load(f)
    comparison_data.append({
        'Run ID': r['meta']['run_id'],
        'Status': r['meta']['overall_status'],
        'Success': f"{r['meta']['success_count']}/{r['meta']['total_notebooks']}",
        'Duration (min)': f"{r['meta']['duration_seconds']/60:.1f}"
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))


📊 Comparing Last 3 Runs:

         Run ID  Status Success Duration (min)
20251223_183617  FAILED    1/22            6.4
20251223_200719  FAILED    2/24           18.4
20251224_205625 SUCCESS    3/26            6.1
